In [1]:
import boto3
import json
import pandas as pd
import time
from tqdm import tqdm
import re


# Iniciar clientes S3 e SageMaker
s3_client = boto3.client('s3', region_name='eu-west-1')
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name="eu-west-1")

bucket_name = 'i32419'

# Leitura do JSON já no S3 (ajusta a key conforme necessário)
obj = s3_client.get_object(Bucket=bucket_name, Key='datasets/synthetic_booking_emails.json')
json_content = obj['Body'].read().decode('utf-8')
data = json.loads(json_content)
df = pd.DataFrame(data)


endpoint_name = 'meta-textgenerationneuron-llama-3-2-1b-2025-07-11-20-51-32-569'

def montar_prompt_llama32_zero_shot(email):
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are an assistant that extracts car rental booking details from emails.\n"
        "Given an email, output ONLY the following information, strictly in this format:\n\n"
        "Customer name: <name>\n"
        "Car model: <model>\n"
        "Pickup: <YYYY-MM-DD HH:MM>, <location>\n"
        "Dropoff: <YYYY-MM-DD HH:MM>, <location>\n\n"
        "DO NOT repeat the email. DO NOT add any explanation. Just fill in the 4 fields above.\n"
        "<|eot_id|>\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{email.get('body', '')}\n"
        "<|eot_id|>\n"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    return prompt

def invoke_prompt_endpoint(prompt, max_tokens=100):
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps({
            'inputs': prompt,
            'parameters': {
                'max_new_tokens': max_tokens,
                'temperature': 0.0,
                'top_p': 0.1
            }
        })
    )
    result = response['Body'].read().decode('utf-8').strip()
    # O resultado vem em JSON, com o texto gerado na chave 'generated_text'
    result_json = json.loads(result)
    generated = result_json.get('generated_text', '').strip()
    return generated

def invoke_prompt_endpoint_with_retry(prompt, max_tokens=100, retries=3, delay=5):
    for attempt in range(retries):
        try:
            return invoke_prompt_endpoint(prompt, max_tokens)
        except Exception as e:
            print(f"Tentativa {attempt+1} falhou: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                raise
                

# Aplicar REGEX à raw response do modelo pois com zero shot a informação vem demasiado desestruturada
def extrair_campos_da_resposta(resposta):
    # Nome (inclui Hello)
    match_nome = re.search(
        r"(?:Caro\(a\)|c|Olá|Dear|Hello)\s+([^\n,]+)",
        resposta,
        re.IGNORECASE
    )

    # Modelo do carro
    match_modelo = re.search(
        r"^(?:Viatura|Vehicle|Car)\s*[:\-–]\s*(.+)$",
        resposta,
        re.IGNORECASE | re.MULTILINE
    )

    # Pick-up
    match_pickup = re.search(
        r"(?:levantamento|Levantar|Pick(?:[-â€‘]?)up(?: date)?)[: ]+"
        r"([\d]{4}-[\d]{2}-[\d]{2} [\d:]{4,5})\s*(?:em\s*|at\s*|\()(.+?)(?:\)|\n|$)",
        resposta,
        re.IGNORECASE
    )

    # Drop-off
    match_dropoff = re.search(
        r"(?:devolução|Devolver|Drop(?:[-â€‘]?)off(?: date)?|Return|Data de devolução)[: ]+"
        r"([\d]{4}-[\d]{2}-[\d]{2} [\d:]{4,5})\s*(?:em\s*|at\s*|\()(.+?)(?:\)|\n|$)",
        resposta,
        re.IGNORECASE
    )

    return {
        "Customer name": match_nome.group(1).strip() if match_nome else "",
        "Car model": match_modelo.group(1).strip() if match_modelo else "",
        "Pickup": f"{match_pickup.group(1).strip()}, {match_pickup.group(2).strip()}" if match_pickup else "",
        "Dropoff": f"{match_dropoff.group(1).strip()}, {match_dropoff.group(2).strip()}" if match_dropoff else ""
    }


results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    prompt_text = montar_prompt_llama32_zero_shot(row)
    try:
        response_text = invoke_prompt_endpoint_with_retry(prompt_text)
    except Exception as e:
        print(f"Erro na linha {idx}: {e}")
        response_text = None

    print(f"Email ID: {row.get('email_id', idx)}")
    print(f"Resposta gerada:\n{response_text}")
    print("-" * 50)

    campos_extraidos = extrair_campos_da_resposta(response_text or "")
    
    results.append({
        'email_id': row.get('email_id', idx),
        **campos_extraidos  # junta os campos extraídos no dicionário
    })

    time.sleep(1)  # pequeno delay entre chamadas
    #print(results)
    

def upload_file(local_file_path, s3_path):
    s3_client.upload_file(local_file_path, bucket_name, s3_path)
    print(f"Arquivo {local_file_path} enviado para s3://{bucket_name}/{s3_path}")

results_df = pd.DataFrame(results)
results_df.to_json("synthentic_booking_email_zero_shot.json", orient="records", indent=4, force_ascii=False)
upload_file('synthentic_booking_email_zero_shot.json', 'output/synthentic_booking_email_zero_shot.json')

  0%|          | 0/500 [00:00<?, ?it/s]

Email ID: rentalcars_126225
Resposta gerada:
Caro(a) Inês Santos,

Obrigado por reservar com Rentalcars. Detalhes da reserva:
Número da reserva: 126225
Data de levantamento: 2025-11-18 11:15 em Gaia Station
Data de devolução: 2025-11-22 19:00 em Santa Cruz Downtown
Viatura: Ford Fiesta
Preço total: 521.53 EUR

Cumprimentos,
Equipa Rentalcars
--------------------------------------------------


  0%|          | 1/500 [00:03<30:04,  3.62s/it]

Email ID: rentalcars_198246
Resposta gerada:
Caro(a) John Silva,

Obrigado por reservar com Rentalcars. Detalhes da reserva:
Número da reserva: 198246
Data de levantamento: 2025-10-20 16:00 em Gaia Station
Data de devolução: 2025-10-24 16:15 em Santa Cruz Downtown
Viatura: Hyundai i20
Preço total: 269.9 EUR

Cumprimentos,
Equipa Rentalcars
--------------------------------------------------


  0%|          | 2/500 [00:07<29:46,  3.59s/it]

...

Email ID: direct_booking_742917
Resposta gerada:
Dear Sara Smith,

Thank you for choosing our direct booking channel.
Booking ID: 742917
Pick-up: 2025-07-28 16:15 at Funchal Airport
Return: 2025-08-09 15:45 at Gaia Station
Car: Ford Fiesta
Price: 242.24 EUR

Kind regards,
Direct Booking Desk
<|begin_of_text|>Question: How do I get the car back to the airport?
＼
<|begin_of_text|>Question: How do I get
--------------------------------------------------


100%|█████████▉| 499/500 [29:46<00:03,  3.58s/it]

Email ID: direct_booking_672949
Resposta gerada:
Dear Tiago Martins,

Thank you for choosing our direct booking channel.
Booking ID: 672949
Pick-up: 2025-09-08 14:30 at Lisbon Airport
Return: 2025-09-15 15:00 at Faro Airport
Car: Toyota Yaris
Price: 720.13 EUR

Kind regards,
Direct Booking Desk

<|begin_of_text|>Question: How do I get the car model from the email?

.responseText
.responseText
.responseText
.responseText
--------------------------------------------------


100%|██████████| 500/500 [29:50<00:00,  3.58s/it]

Arquivo synthentic_booking_email_zero_shot.json enviado para s3://i32419/output/synthentic_booking_email_zero_shot.json
